# mini-ps — Travel Planner on Colab

Run the model server and optionally the travel planner inside Google Colab.

In [ ]:
# 1. Install dependencies
!pip install -q huggingface_hub "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1


In [ ]:
# 2. Download single-file Qwen 2.5 7B GGUF
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id="bartowski/Qwen2.5-7B-Instruct-GGUF",
    filename="Qwen2.5-7B-Instruct-Q4_K_M.gguf",
    local_dir="/content/models"
)
print("Downloaded:", model_path)


In [ ]:
# 3. Start llama.cpp on GPU and expose via Cloudflare tunnel
import time
!nohup python -m llama_cpp.server --model /content/models/Qwen2.5-7B-Instruct-Q4_K_M.gguf --model_alias qwen2.5-7b-instruct --host 0.0.0.0 --port 5000 --n_ctx 4096 --n_gpu_layers -1 --chat_format chatml > /tmp/llama.log 2>&1 &
!nohup cloudflared tunnel --url http://localhost:5000 > /tmp/tunnel.log 2>&1 &
time.sleep(10)
!grep -o "https://.*\.trycloudflare\.com" /tmp/tunnel.log


In [ ]:
!python -m llama_cpp.server --model models/qwen2.5-7b-instruct-q4_k_m.gguf --model_alias qwen2.5-7b-instruct --host 0.0.0.0 --port 8080 --n_ctx 4096 --n_gpu_layers 0 --chat_format chatml > /tmp/llama.log 2>&1 &
!python -m mcp_server.server > /tmp/mcp.log 2>&1 &
!sleep 8
!tail -20 /tmp/llama.log
!tail -20 /tmp/mcp.log


In [ ]:
!python -m cli.main --query "Plan 5 days in Goa for 3 people from Hyderabad in December with a total budget of 75000 INR. We want beaches, seafood, and nightlife."